

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv("../data/train.csv")
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# ==============================================
# 1. Missing Value Handling
# ==============================================
# Check missing values
print("Missing values before handling:")
print(df.isnull().sum())

# Age: Impute with Median
df['Age'].fillna(df['Age'].median(), inplace=True)

# Embarked: Impute with Mode
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Cabin: Extract Deck information
df['Deck'] = df['Cabin'].apply(lambda x: str(x)[0] if pd.notna(x) else 'U')
df.drop('Cabin', axis=1, inplace=True)

print("\n✅ Missing values handled.")

In [ ]:
# ==============================================
# 2. Outlier Handling
# ==============================================

# Function to cap outliers using IQR method
def cap_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[column] = np.where(df[column] < lower_bound, lower_bound, df[column])
    df[column] = np.where(df[column] > upper_bound, upper_bound, df[column])

cap_outliers(df, 'Fare')
cap_outliers(df, 'Age')

print("✅ Outliers capped.")

In [ ]:
# ==============================================
# 3. Data Consistency
# ==============================================

# Remove duplicates
df.drop_duplicates(inplace=True)

# Standardize text
df['Sex'] = df['Sex'].str.lower()

print("✅ Data cleaned and consistent.")

# Save cleaned dataset
df.to_csv("../data/train_cleaned.csv", index=False)
print("✅ Cleaned data saved as train_cleaned.csv")

In [ ]:
# ==============================================
# Create Derived Features
# ==============================================

# 1. Family Size
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# 2. Is Alone
df['IsAlone'] = 0
df.loc[df['FamilySize'] == 1, 'IsAlone'] = 1

# 3. Title Extraction from Name
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Group rare titles
rare_titles = ['Dr', 'Rev', 'Col', 'Major', 'Lady', 'Sir', 'Capt', 'Countess', 'Don', 'Jonkheer', 'Dona']
df['Title'] = df['Title'].replace(rare_titles, 'Rare')
df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')

# 4. Age Groups
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100], 
                        labels=['Child', 'Teen', 'Adult', 'Senior', 'Elderly'])

# 5. Fare per Person
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

print("✅ All new features created.")
df.head()

In [ ]:
# ==============================================
# Encoding Categorical Variables
# ==============================================

# One-Hot Encoding
df = pd.get_dummies(df, columns=['Sex', 'Embarked', 'Title', 'Deck', 'AgeGroup'], drop_first=True)

print("✅ Encoding completed.")
print(f"New shape: {df.shape}")

In [ ]:
# ==============================================
# Feature Transformation
# ==============================================

# Log transform for skewed data
df['Fare_log'] = np.log1p(df['Fare'])

# Standardization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df[['Age', 'Fare']] = scaler.fit_transform(df[['Age', 'Fare']])

print("✅ Transformations applied.")

In [ ]:
# ==============================================
# 1. Correlation Analysis
# ==============================================

# Select only numerical features
numeric_df = df.select_dtypes(include=[np.number])

# Plot heatmap
plt.figure(figsize=(15, 10))
sns.heatmap(numeric_df.corr(), annot=False, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

# Remove highly correlated features (> 0.9)
corr_matrix = numeric_df.corr()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
df.drop(to_drop, axis=1, inplace=True)

print(f"✅ Dropped {len(to_drop)} highly correlated features.")

In [ ]:
# ==============================================
# 2. Feature Importance using Random Forest
# ==============================================

from sklearn.ensemble import RandomForestClassifier

# Prepare data
X = df.drop(['PassengerId', 'Name', 'Ticket', 'Survived'], axis=1, errors='ignore')
y = df['Survived']

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# Show importance
importances = pd.Series(model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

print("✅ Feature Importance Ranking:")
print(importances.head(10))

# Plot
plt.figure(figsize=(12, 8))
importances.head(10).plot(kind='barh', color='skyblue')
plt.title('Top 10 Important Features')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# ==============================================
# Selected Features
# ==============================================

# Keep only important features
selected_features = importances[importances > 0.02].index.tolist()
print("✅ Final Selected Features:")
for feat in selected_features:
    print(" -", feat)

# Save final processed data
df[selected_features + ['Survived']].to_csv("../data/train_final.csv", index=False)
print("\n✅ Final dataset saved.")